# 08 · Feature engineering: how do we represent an *unseen* perturbation?
A model cannot learn from a one-hot gene ID to predict a gene it never saw. We therefore describe each perturbed gene by **properties that exist without any perturbation data for that gene**:
- **Co-expression embedding** from *control cells only* (genes with similar expression programs get similar vectors). Leakage-safe: no perturbation outcome is used.
- **Pathway/GO membership** compressed with SVD (external knowledge; needs internet once).

Features are indexed by *gene*, so any gene in the expression matrix can be encoded, including genes that were never perturbed.

In [ ]:
# ---- Bootstrap: Colab (Google Drive) or local ----
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/PerturbAI.git"   # <-- edit once after you create the repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/PerturbAI"):
        subprocess.run(["git", "clone", REPO_URL, "/content/PerturbAI"], check=True)
    os.chdir("/content/PerturbAI")
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
else:
    try:
        import perturbai  # installed with `pip install -e .`
    except ImportError:
        sys.path.insert(0, os.path.abspath("../src"))  # fallback when running from notebooks/
from perturbai.paths import get_paths
P = get_paths()
print("Data root:", P.root)

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, matplotlib.pyplot as plt
from perturbai.paths import artifact
from perturbai.utils import set_seed, load_config
CFG = load_config("data_config.yaml"); MCFG = load_config("model_config.yaml")
set_seed(CFG["seed"])
SYNTHETIC = False        # True = 1-minute smoke test on simulated data (no download)
DS = "replogle_k562_essential"
INFO = CFG["datasets"][DS]
PERT, CTRL, BATCH = INFO["perturbation_col"], INFO["control_label"], INFO["batch_col"]
TAG = DS + ("_synthetic" if SYNTHETIC else "")   # artifact prefix

In [ ]:
import scanpy as sc
from perturbai.features.embeddings import coexpression_embedding, pathway_membership_matrix
from sklearn.decomposition import TruncatedSVD
adata = sc.read_h5ad(P.processed / artifact("subset_adata", TAG))
ctrl = adata[adata.obs[PERT].astype(str) == CTRL]
emb = coexpression_embedding(ctrl.X, adata.var_names, MCFG["features"]["coexpr_components"] if not SYNTHETIC else 16)
print("co-expression embedding:", emb.shape)

In [ ]:
feats = emb.copy()
try:
    import gseapy as gp
    lib = gp.get_library("GO_Biological_Process_2023")
    M = pathway_membership_matrix(lib, adata.var_names, min_size=5, max_size=300)
    k = min(32, M.shape[1] - 1)
    go = pd.DataFrame(TruncatedSVD(k, random_state=0).fit_transform(M.values), index=M.index,
                      columns=[f"go_{i}" for i in range(k)])
    feats = feats.join(go); print("GO features:", go.shape)
except Exception as e:
    print("GO features skipped (offline?):", e)
feats.to_parquet(P.processed / artifact("features", TAG)); feats.shape

In [ ]:
# Sanity: do genes in the same complex end up close in embedding space? (look at nearest neighbours of a gene)
E = emb.values / (np.linalg.norm(emb.values, axis=1, keepdims=True) + 1e-9)
g0 = emb.index[0]; sims = pd.Series(E @ E[0], index=emb.index).sort_values(ascending=False)
print(g0, "->", list(sims.index[1:11]))
delta = pd.read_parquet(P.processed / artifact("effects", TAG))
print("perturbations with features:", delta.index.isin(emb.index).sum(), "of", len(delta))

**Leakage check:** embeddings use control cells + external annotations only. If you later add features derived from perturbed cells, compute them *inside* each training split.